# Simple Neural Network (Multilayer Preceptron)

In [382]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.decomposition import PCA
import mlflow

In [383]:
train_df = pd.read_csv('../train_competition_2026.csv')
train_df.head()

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4,y_1,y_2
0,0,0,2068-09-19 23:34:11,1.38,49,7,1,3,1,0,1,105.5,95.0,67.4,36.6,23.2,33.4,107.4
1,0,0,2068-09-19 23:35:11,1.38,49,7,1,3,1,0,1,104.4,95.0,66.4,37.8,22.7,33.4,107.4
2,0,0,2068-09-19 23:36:11,1.38,49,7,1,3,1,0,1,104.0,95.0,65.2,37.0,22.1,33.4,107.4
3,0,0,2068-09-19 23:37:11,1.38,49,7,1,3,1,0,1,102.8,95.0,63.4,35.9,20.7,33.4,107.4
4,0,0,2068-09-19 23:38:11,1.38,49,7,1,3,1,0,1,101.3,95.1,59.1,34.5,18.1,33.4,107.4


In [384]:
train_df['time'] = pd.to_datetime(train_df['time'], format='%Y-%m-%d %H:%M:%S')

In [385]:
test_df = pd.read_csv('../test_no_outcome.csv')
test_df.head()

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4
0,18,1,2134-04-01 22:23:14,-1.0,38,1,1,1,0,0,0,105.4,99.8,50.7,61.4,36.8
1,18,1,2134-04-01 22:24:14,-1.0,38,1,1,1,0,0,0,105.4,99.4,49.4,61.1,36.2
2,18,1,2134-04-01 22:25:14,-1.0,38,1,1,1,0,0,0,104.6,99.0,49.7,61.4,36.6
3,18,1,2134-04-01 22:26:14,-1.0,38,1,1,1,0,0,0,104.5,99.6,51.7,61.8,37.2
4,18,1,2134-04-01 22:27:14,-1.0,38,1,1,1,0,0,0,104.6,99.5,52.5,61.9,37.5


## MLP with PCA

In [386]:
def flatten_blocks(df, feature_cols):
    X_flat = df.groupby(["obs"])[feature_cols].apply(lambda block: block.values.flatten())
    X_flat = pd.DataFrame(X_flat.tolist(), index=X_flat.index)
    return X_flat.reset_index()

In [387]:
feature_cols = [c for c in train_df.columns if c not in ["sub_id", "obs", "time", "y_1", "y_2"]]

X_train_flat = flatten_blocks(train_df, feature_cols)
X_test_flat  = flatten_blocks(test_df, feature_cols)

In [388]:
X_train_flat.head()

,obs,0,1,2,3,4,5,6,7,8,...,380,381,382,383,384,385,386,387,388,389
0,0,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,105.5,...,1.0,3.0,1.0,0.0,1.0,94.5,93.8,59.8,34.3,17.7
1,1,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,98.6,...,1.0,3.0,1.0,0.0,1.0,95.3,92.9,52.1,38.0,22.0
2,2,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,93.1,...,1.0,3.0,1.0,0.0,1.0,91.2,93.4,52.5,36.1,19.1
3,3,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,85.1,...,1.0,3.0,1.0,0.0,1.0,87.1,95.0,61.8,36.4,20.8
4,4,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,88.7,...,1.0,3.0,1.0,0.0,1.0,86.1,89.6,66.7,32.5,17.8


In [389]:
def extract_targets(df):
    return df.groupby(["obs"])[["y_1", "y_2"]].first().reset_index(drop=True)

y_train = extract_targets(train_df)

In [390]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers=1, hidden_size=32, dropout=0.0):
        super().__init__()
        
        layers = []
        
        # Input layer
        layers.append(nn.Linear(input_dim, hidden_size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        
        # Output layer
        layers.append(nn.Linear(hidden_size, output_dim))
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [398]:
def train_mlp(
    X_train, y_train, X_val, y_val,
    input_dim, output_dim, num_layers=1,
    hidden_size=32, dropout=0.0, weight_decay=1e-4,
    epochs=500, epoch_int=10, patience=20, lr=1e-3,
):
    
    model = SimpleMLP(
        input_dim=input_dim,
        output_dim=output_dim,
        num_layers=num_layers,
        hidden_size=hidden_size,
        dropout=dropout
    ).to("cpu")

    criterion = nn.MSELoss()  # change for classification if needed
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32).to("cpu")
    y_train = torch.tensor(y_train, dtype=torch.float32).to("cpu")
    X_val   = torch.tensor(X_val, dtype=torch.float32).to("cpu")
    y_val   = torch.tensor(y_val, dtype=torch.float32).to("cpu")

    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()

        train_pred = model(X_train)
        train_loss = criterion(train_pred, y_train)

        train_loss.backward()
        optimizer.step()

        # Val
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val)
            
            # Calc r2
            train_r2 = r2_score(y_train.cpu().numpy(), train_pred.detach().cpu().numpy())
            val_r2 = r2_score(y_val.cpu().numpy(), val_pred.cpu().numpy())

        if epoch % epoch_int == 0:
            print(f'Epoch {epoch:03d} | '
                f'Train Loss: {train_loss.item():.4f} | Val Loss {val_loss.item():.4f} | '
                f'Train R2: {train_r2:.4f} | Val R2 {val_r2:.4f}')

        # Early stopping
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # Restore best model
    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        final_val_pred = model(X_val)
        final_val_loss = criterion(final_val_pred, y_val)

    return model, final_val_loss

In [399]:
def run_pipeline(hp):
    k = hp['pca_k']
    
    from sklearn.preprocessing import StandardScaler
    
    scaler_x = StandardScaler()
    X_train_raw = X_train_flat.drop(columns=["obs"])
    
    X_train_scaled = scaler_x.fit_transform(X_train_raw)

    pca = PCA(n_components=k, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train)

    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train_pca, y_train_scaled, test_size=0.2, random_state=42
    )

    model, val_loss = train_mlp(
        X_train_split, y_train_split, X_val, y_val,
        input_dim=X_train_split.shape[1],
        output_dim=2,
        num_layers=hp['num_layers'],
        hidden_size=hp['hidden_size'],
        dropout=hp['dropout'],
        weight_decay=hp['weight_decay'],
        epochs=hp['epochs'],
        epoch_int=hp['epoch_int'],
        patience=hp['patience'],
        lr=hp['lr']
    )
    
    return model, val_loss, scaler_x, scaler_y, pca

In [ ]:
hp = {
    'pca_k': 50,
    'hidden_size': 64,
    'num_layers': 4,
    'dropout': 0.0,
    'weight_decay': 1e-3,
    'epochs': 10000, 
    'epoch_int': 50,
    'patience': 50,
    'lr': 1e-3
}

model, val_loss, scaler_x, scaler_y, pca = run_pipeline(hp)

Epoch 000 | Train Loss: 1.0114 | Val Loss 0.9839 | Train R2: -0.0062 | Val R2 -0.0050
Epoch 050 | Train Loss: 0.8845 | Val Loss 0.8622 | Train R2: 0.1202 | Val R2 0.1189
Epoch 100 | Train Loss: 0.6308 | Val Loss 0.6131 | Train R2: 0.3727 | Val R2 0.3727
Epoch 150 | Train Loss: 0.3173 | Val Loss 0.3107 | Train R2: 0.6842 | Val R2 0.6833
Epoch 200 | Train Loss: 0.2398 | Val Loss 0.2406 | Train R2: 0.7611 | Val R2 0.7553
Epoch 250 | Train Loss: 0.2241 | Val Loss 0.2257 | Train R2: 0.7767 | Val R2 0.7705
Epoch 300 | Train Loss: 0.2152 | Val Loss 0.2173 | Train R2: 0.7856 | Val R2 0.7791
Epoch 350 | Train Loss: 0.2093 | Val Loss 0.2117 | Train R2: 0.7914 | Val R2 0.7848
Epoch 400 | Train Loss: 0.2051 | Val Loss 0.2079 | Train R2: 0.7957 | Val R2 0.7887
Epoch 450 | Train Loss: 0.2017 | Val Loss 0.2051 | Train R2: 0.7991 | Val R2 0.7916
Epoch 500 | Train Loss: 0.1989 | Val Loss 0.2029 | Train R2: 0.8018 | Val R2 0.7938
Epoch 550 | Train Loss: 0.1965 | Val Loss 0.2013 | Train R2: 0.8042 | Val 

In [405]:
def get_real_predictions(model, X_test_df, scaler_x, scaler_y, pca):
    X_raw = X_test_df.drop(columns=["obs"]) if "obs" in X_test_df.columns else X_test_df
    X_scaled = scaler_x.transform(X_raw)
    
    X_pca = pca.transform(X_scaled)
    
    model.eval()
    X_tensor = torch.FloatTensor(X_pca)
    
    with torch.no_grad():
        y_pred_tensor = model(X_tensor)

    y_pred_scaled = y_pred_tensor.cpu().numpy()
    y_pred_final = scaler_y.inverse_transform(y_pred_scaled)
    
    return y_pred_final

In [406]:
predictions = get_real_predictions(model, X_test_flat, scaler_x, scaler_y, pca)

y_test_pred = get_real_predictions(model, X_test_flat, scaler_x, scaler_y, pca)

pred_df = pd.concat(
    [
        X_test_flat[["obs"]].reset_index(drop=True), 
        pd.DataFrame(y_test_pred, columns=["y_1", "y_2"])
    ],
    axis=1
)

len(y_test_pred)

3450

In [407]:
pred_df.to_csv("mlp_pca_1.csv", index=False)

In [397]:
def run_with_mlflow(hp):
    run_name = f"lstm_k{hp['pca_k']}_h{hp['hidden_size']}_l{hp['num_layers']}_d{hp['dropout']}_w{hp['weight_decay']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(hp)
        model, val_loss, _, _, _ = run_pipeline(hp)
        mlflow.log_metric("val_loss", val_loss)
        
        return model, val_loss

if __name__ == '__main__':
    db_path = "/Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/advanced_ml_project/mlflow.db"
    mlflow.set_tracking_uri(f"sqlite:///{db_path}")
    mlflow.set_experiment("AML_Project_MLP_Grid_Search")
    
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"Active Experiment: {mlflow.get_experiment_by_name('AML_Project_MLP_Grid_Search').experiment_id}")

    from itertools import product

    best_score = float("inf")
    
    pca_ks = [10, 30, 50]
    hidden_sizes = [8, 16, 32, 64]
    num_layers = [1, 2, 3, 4]
    dropouts = [0.0, 0.2, 0.4]
    weight_decays = [1e-5, 1e-4, 1e-3]
    
    best_model = None
    best_pca = None

    for ks, hs, nl, dr, wd in product(pca_ks, hidden_sizes, num_layers, dropouts, weight_decays):
        
        hp = {
            'pca_k': ks,
            'hidden_size': hs,
            'num_layers': nl,
            'dropout': dr,
            'weight_decay': wd,
            'epochs': 10000, 
            'epoch_int': 50,
            'patience': 50,
            'lr': 5e-3
        }
        
        model, score = run_with_mlflow(hp)
        
        if score < best_score:
            best_score = score
            best_hp = hp
            best_model = model
            
            best_pca = PCA(n_components=hp['pca_k'], random_state=42)
            best_pca.fit(X_train_flat.drop(columns=["obs"]))
            

    print("Best:", best_hp, best_score)

Tracking URI: sqlite:////Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/advanced_ml_project/mlflow.db
Active Experiment: 1
Early stopping at epoch 810
Early stopping at epoch 661
Early stopping at epoch 665
Early stopping at epoch 576
Early stopping at epoch 678
Early stopping at epoch 640
Early stopping at epoch 400
Early stopping at epoch 400
Early stopping at epoch 506
Early stopping at epoch 563
Early stopping at epoch 816
Early stopping at epoch 334
Early stopping at epoch 339
Early stopping at epoch 293
Early stopping at epoch 266
Early stopping at epoch 378
Early stopping at epoch 277
Early stopping at epoch 334
Early stopping at epoch 508
Early stopping at epoch 321
Early stopping at epoch 412
Early stopping at epoch 262
Early stopping at epoch 223
Early stopping at epoch 374
Early stopping at epoch 407
Early stopping at epoch 368
Early stopping at epoch 353
Early stopping at epoch 467
Early stopping at epoch 291
Early stopping at epoch 428
Early stopping at epoch 218
Early